In [18]:
import pandas as pd

# load csv
df = pd.read_csv("testimonies.csv")

# US states for comparison
states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire",
    "New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
    "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia",
    "Wisconsin","Wyoming"
]

# keyword matching
def extract_states(text):
    found = []
    for state in states:
        if isinstance(text, str) and state.lower() in text.lower():
            found.append(state)
    return list(set(found))  # unique

df["states_found"] = df["testimony"].apply(extract_states)

# select needed columns and person can have multiple rows depending on count of states mentioned
df_states = df[["lname", "fname", "state", "states_found"]].explode("states_found")

# preview results
df_states.head()

,lname,fname,state,states_found
0,Akers,Diane,Washington,Iowa
1,Anderson,June M.,Alaska,Missouri
1,Anderson,June M.,Alaska,Alaska
2,Anderson,Wanda A.,Pennsylvania,Pennsylvania
3,Armstrong,Maxine Webb,Missouri,Iowa


In [19]:
import sqlite3
DB_PATH = "testimonies.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            print(rows)
        else:
            conn.commit()
            print("OK")
    except sqlite3.Error as err:
        print(f"SQLite error: {err}")

df_states.to_sql("states_found_table", conn, if_exists="replace", index=False)

394

In [20]:
# count of people from different states
run_sql("""
SELECT state, COUNT(*) as count
FROM states_found_table
GROUP BY state
ORDER BY count DESC;
""")

[('Missouri', 56), ('Michigan', 44), ('Iowa', 37), ('California', 26), ('Washington', 19), ('Florida', 19), ('Oregon', 16), ('Oklahoma', 16), ('Illinois', 16), ('Texas', 14), ('Kansas', 14), ('Arizona', 12), ('Ohio', 10), ('Montana', 10), ('Alabama', 10), ('Colorado', 9), ('Saskatchewan', 7), ('Ontario', 6), ('New Mexico', 6), ('Mississippi ', 5), ('Minnesota', 5), ('Alaska', 5), ('Alberta', 4), ('Maryland', 3), ('Louisiana', 3), ('Arkansas', 3), ('Wyoming', 2), ('Wisconsin', 2), ('West Virginia', 2), ('Tennessee', 2), ('Indiana', 2), ('Idaho', 2), ('Warwickshire', 1), ('Pennsylvania', 1), ('Maine', 1), ('Georgia', 1), ('British Columbia', 1), ('Baden-Württemberg', 1), ('Alberta ', 1)]


In [21]:
# count of mentions of different states
run_sql("""
SELECT states_found, COUNT(*) as count
FROM states_found_table
GROUP BY states_found
ORDER BY count DESC;
""")

[(None, 44), ('Missouri', 41), ('Iowa', 37), ('Michigan', 28), ('Kansas', 27), ('Maine', 20), ('California', 20), ('Florida', 16), ('Washington', 12), ('Texas', 12), ('Arizona', 11), ('Oklahoma', 9), ('Montana', 9), ('Oregon', 8), ('Mississippi', 8), ('Illinois', 7), ('Colorado', 7), ('Wyoming', 6), ('Ohio', 6), ('Minnesota', 6), ('Indiana', 6), ('Alabama', 6), ('Wisconsin', 4), ('Virginia', 4), ('Nebraska', 4), ('Maryland', 4), ('Idaho', 4), ('Alaska', 4), ('Pennsylvania', 3), ('New York', 3), ('Louisiana', 3), ('Hawaii', 3), ('Utah', 2), ('North Dakota', 2), ('New Mexico', 2), ('Arkansas', 2), ('West Virginia', 1), ('Tennessee', 1), ('Kentucky', 1), ('Georgia', 1)]
